In [1]:
"""
# 🏦 CreditSense: Loan Risk Assessment — Baseline Template

## Your Two Tasks

**Task A — Classification:** Predict `RiskTier` (0 to 4)
- 0 = Very Low Risk → 4 = Very High Risk

**Task B — Regression:** Predict `InterestRate` (%)
- Annual percentage rate offered to the applicant
- Range: 4.99% (best) to 35.99% (worst)

## What this baseline does
- Loads and preprocesses the data
- Trains a **Logistic Regression** for Task A (~53% accuracy)
- Trains a **Linear Regression** for Task B (R² ~0.50)
- Saves a submission file

**Your goal: beat both baselines!**
"""

'\n# 🏦 CreditSense: Loan Risk Assessment — Baseline Template\n\n## Your Two Tasks\n\n**Task A — Classification:** Predict `RiskTier` (0 to 4)\n- 0 = Very Low Risk → 4 = Very High Risk\n\n**Task B — Regression:** Predict `InterestRate` (%)\n- Annual percentage rate offered to the applicant\n- Range: 4.99% (best) to 35.99% (worst)\n\n## What this baseline does\n- Loads and preprocesses the data\n- Trains a **Logistic Regression** for Task A (~53% accuracy)\n- Trains a **Linear Regression** for Task B (R² ~0.50)\n- Saves a submission file\n\n**Your goal: beat both baselines!**\n'

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             r2_score, mean_squared_error)
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [2]:
train = pd.read_csv('creditsense-ai1215/credit_train.csv')

X     = train.drop(['RiskTier', 'InterestRate'], axis=1)
y_cls = train['RiskTier']
y_reg = train['InterestRate']

print(f"Dataset shape: {X.shape}")
print(f"\nTask A — RiskTier distribution:\n{y_cls.value_counts().sort_index()}")
print(f"\nTask B — InterestRate summary:\n{y_reg.describe().round(2)}")

# TODO: Explore the data further
# X.isnull().sum()       ← check missing values
# X.describe()           ← check feature ranges
# X.dtypes               ← check feature types

Dataset shape: (35000, 55)

Task A — RiskTier distribution:
RiskTier
0    6724
1    7283
2    6998
3    6812
4    7183
Name: count, dtype: int64

Task B — InterestRate summary:
count    35000.00
mean         7.31
std          4.19
min          4.99
25%          4.99
50%          6.08
75%          7.94
max         35.99
Name: InterestRate, dtype: float64


In [3]:
def simple_preprocess(X_train, X_test=None):
    """
    Fill missing values and encode categorical features.

    TODO: Improve this!
    - Add is_missing indicator columns before filling
    - Try different imputation strategies
    - Create new features from existing ones
    """
    X_train = X_train.copy()
    if X_test is not None:
        X_test = X_test.copy()

    num_cols = X_train.select_dtypes(include=[np.number]).columns
    cat_cols = X_train.select_dtypes(include=['object']).columns

    print(f"Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}")

    # Numeric → fill with median
    for col in num_cols:
        med = X_train[col].median()
        X_train[col] = X_train[col].fillna(med)
        if X_test is not None:
            X_test[col] = X_test[col].fillna(med)

    # Categorical → fill with mode, then label encode
    for col in cat_cols:
        mode = X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else 'Unknown'
        X_train[col] = X_train[col].fillna(mode)
        if X_test is not None:
            X_test[col] = X_test[col].fillna(mode)

        le = LabelEncoder()
        if X_test is not None:
            combined = pd.concat([X_train[col].astype(str), X_test[col].astype(str)])
            le.fit(combined)
            X_train[col] = le.transform(X_train[col].astype(str))
            X_test[col]  = le.transform(X_test[col].astype(str))
        else:
            X_train[col] = le.fit_transform(X_train[col].astype(str))

    X_train = X_train.fillna(0)
    if X_test is not None:
        X_test = X_test.fillna(0)

    if X_test is not None:
        return X_train, X_test
    return X_train


X_processed = simple_preprocess(X)
print(f"\nAfter preprocessing — shape: {X_processed.shape} | missing: {X_processed.isnull().sum().sum()}")

Numeric: 46 | Categorical: 9

After preprocessing — shape: (35000, 55) | missing: 0


In [4]:
X_train, X_val, y_train_cls, y_val_cls, y_train_reg, y_val_reg = train_test_split(
    X_processed, y_cls, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

print(f"Train: {X_train.shape} | Val: {X_val.shape}")

scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_va_sc = scaler.transform(X_val)

Train: (28000, 55) | Val: (7000, 55)


In [5]:
print("=" * 55)
print("TASK A: RiskTier Classification — Logistic Regression")
print("=" * 55)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_tr_sc, y_train_cls)

y_pred_cls = clf.predict(X_va_sc)

print(f"Train Accuracy: {accuracy_score(y_train_cls, clf.predict(X_tr_sc))*100:.2f}%")
print(f"Val   Accuracy: {accuracy_score(y_val_cls, y_pred_cls)*100:.2f}%")
print(f"\nClassification Report:")
print(classification_report(
    y_val_cls, y_pred_cls,
    target_names=['VeryLow(0)','Low(1)','Moderate(2)','High(3)','VeryHigh(4)']
))

# TODO: Try RandomForestClassifier, XGBClassifier

TASK A: RiskTier Classification — Logistic Regression
Train Accuracy: 64.50%
Val   Accuracy: 63.43%

Classification Report:
              precision    recall  f1-score   support

  VeryLow(0)       0.60      0.64      0.62      1345
      Low(1)       0.45      0.52      0.48      1456
 Moderate(2)       0.49      0.37      0.42      1400
     High(3)       0.72      0.75      0.73      1362
 VeryHigh(4)       0.92      0.90      0.91      1437

    accuracy                           0.63      7000
   macro avg       0.63      0.63      0.63      7000
weighted avg       0.63      0.63      0.63      7000



In [ ]:
print("=" * 55)
print("TASK B: InterestRate Regression — Linear Regression")
print("=" * 55)

reg = LinearRegression()
reg.fit(X_tr_sc, y_train_reg)

y_pred_reg = reg.predict(X_va_sc)
val_r2     = r2_score(y_val_reg, y_pred_reg)
val_rmse   = mean_squared_error(y_val_reg, y_pred_reg) ** 0.5

print(f"Train R²:  {r2_score(y_train_reg, reg.predict(X_tr_sc)):.4f}")
print(f"Val   R²:  {val_r2:.4f}")
print(f"Val RMSE:  {val_rmse:.4f}%")
print(f"Val MAE:   {np.mean(np.abs(y_val_reg - y_pred_reg)):.4f}%")

# TODO: Try RandomForestRegressor, XGBRegressor

TASK B: InterestRate Regression — Linear Regression
Train R²:  0.6211
Val   R²:  0.6263
Val RMSE:  2.6365%
Val MAE:   1.7184%


In [ ]:
test = pd.read_csv('credit_test.csv')

_, X_test_proc = simple_preprocess(X, test)
X_test_scaled  = scaler.transform(X_test_proc)

pred_cls = clf.predict(X_test_scaled)
pred_reg = np.clip(reg.predict(X_test_scaled), 4.99, 35.99).round(2)

submission = pd.DataFrame({
    'Id':           range(len(pred_cls)),
    'RiskTier':     pred_cls,
    'InterestRate': pred_reg,
})
submission.to_csv('submission.csv', index=False)

print("Saved submission.csv")
print(f"\nPreview:\n{submission.head(10).to_string(index=False)}")
print(f"\nRiskTier counts:\n{pd.Series(pred_cls).value_counts().sort_index()}")
print(f"\nInterestRate summary:\n{pd.Series(pred_reg).describe().round(2)}")

Numeric: 46 | Categorical: 9
Saved submission.csv

Preview:
 Id  RiskTier  InterestRate
  0         0          5.69
  1         0          6.27
  2         3          6.61
  3         3          7.50
  4         4         17.69
  5         3          6.49
  6         1          6.36
  7         1          5.54
  8         2          6.36
  9         1          5.61

RiskTier counts:
0    3128
1    3552
2    2233
3    3083
4    3004
Name: count, dtype: int64

InterestRate summary:
count    15000.00
mean         7.36
std          3.38
min          4.99
25%          5.54
50%          6.08
75%          7.35
max         35.99
dtype: float64
